# Bài 40: Phát hiện đối tượng bằng cửa sổ trượt (Object Detection from Sliding Windows)

Mọi bộ phân loại từ đầu đến giờ đều tập trung vào toàn bộ ảnh thay vì các mảng ảnh. Nói cách khác, chúng hỏi "ảnh này *là* gì?" &mdash; giả định rằng ảnh đã chứa đúng một thứ, được căn giữa và cắt gọn.

**Phát hiện (detection)** đối tượng đặt ra một câu hỏi khó hơn: cho một ảnh chứa không, một, hoặc vài đối tượng ở các vị trí và tỉ lệ chưa biết, hãy tìm *vị trí* của từng đối tượng.

Cách tiếp cận kinh điển là **cửa sổ trượt (sliding window)**: huấn luyện một bộ phân loại cho một mảng ảnh kích thước cố định, rồi trượt bộ phân loại qua ảnh để chạy nó ở mọi vị trí (và tỉ lệ). <a href="../references.html#rowley-baluja-kanade-1996">Rowley, Baluja và Kanade (1996)</a> đã làm chính xác điều này với một mạng nơ-ron nhỏ làm bộ phân loại cửa sổ. Ý tưởng này thực sự đi trước thời đại — một trong những ứng dụng thành công đầu tiên của mạng nơ-ron cho một nhiệm vụ thị giác thực tế, hơn một thập kỷ trước sự hồi sinh của học sâu (Bài 37). Bài này xây dựng cách tiếp cận từ đầu đến cuối: một bộ phân loại cửa sổ nhỏ, áp dụng ở mọi vị trí, theo sau là một bước then chốt mà mọi hệ thống thực tế đều cần — gộp các phát hiện trùng lặp.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['apollo11_crew.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## Bước 1: một "khuôn mặt" tổng hợp và một bộ phân loại cửa sổ

Để minh họa cơ chế, không cần dữ liệu thật — chỉ cần một lớp có cấu trúc bên trong nhất quán (đường viền đầu, hai mắt, một cái miệng, luôn theo cùng một bố cục tương đối) so với sự lộn xộn thiếu cấu trúc đó. Huấn luyện một CNN nhỏ như một bộ phân loại cửa sổ thuần túy: cho một vùng cắt kích thước cố định, có khuôn mặt nào lấp đầy nó không, có hay không?

In [ ]:
def make_face(size=16, rng=None):
    img = np.zeros((size, size), dtype=np.float32)
    cy, cx = size // 2, size // 2
    yy, xx = np.mgrid[0:size, 0:size]
    img[((xx - cx) ** 2 + (yy - cy) ** 2) <= (size * 0.42) ** 2] = 0.6       # head
    img[(np.abs(xx - (cx - 3)) <= 1) & (np.abs(yy - (cy - 2)) <= 1)] = 1.0   # left eye
    img[(np.abs(xx - (cx + 3)) <= 1) & (np.abs(yy - (cy - 2)) <= 1)] = 1.0   # right eye
    img[(np.abs(xx - cx) <= 2) & (np.abs(yy - (cy + 3)) <= 1)] = 0.9        # mouth
    if rng is not None:
        img = np.clip(img + rng.normal(0, 0.08, img.shape), 0, 1).astype(np.float32)
    return img

def make_nonface(size=16, rng=None):
    img = rng.uniform(0, 0.5, (size, size)).astype(np.float32)
    if rng.random() < 0.5:
        cy, cx = rng.integers(2, size - 2), rng.integers(2, size - 2)
        r = rng.integers(2, 5)
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2] = rng.uniform(0.4, 0.9)
    return img

rng = np.random.default_rng(11)
N = 200
faces = np.array([make_face(rng=rng) for _ in range(N)])
nonfaces = np.array([make_nonface(rng=rng) for _ in range(N)])
X = np.concatenate([faces, nonfaces])
y = np.concatenate([np.ones(N), np.zeros(N)]).astype(np.float32)
perm = rng.permutation(len(X))
X, y = X[perm], y[perm]
split = int(0.8 * len(X))
Xtr, ytr, Xte, yte = X[:split], y[:split], X[split:], y[split:]

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, im, lbl in zip(axes, list(faces[:3]) + list(nonfaces[:3]), ['face']*3 + ['non-face']*3):
    ax.imshow(im, cmap='gray'); ax.set_title(lbl, fontsize=9); ax.axis('off')
plt.show()

In [ ]:
class WindowClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool2d(1),
        )
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        return self.fc(self.net(x).flatten(1)).squeeze(-1)

torch.manual_seed(0)
model = WindowClassifier()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); yt = torch.tensor(ytr)
for _ in range(300):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(Xt), yt)
    loss.backward()
    opt.step()

with torch.no_grad():
    Xte_t = torch.tensor(Xte).unsqueeze(1)
    acc = ((model(Xte_t) > 0).float() == torch.tensor(yte)).float().mean().item()
print(f'window classifier test accuracy: {acc:.1%}')

## Bước 2: trượt cửa sổ qua một cảnh

Dựng một cảnh lớn hơn chứa hai khuôn mặt ở các vị trí chưa biết cộng với sự lộn xộn của nền, rồi chạy bộ phân loại cửa sổ đã huấn luyện ở mọi vị trí trên một lưới dày đặc (một **cửa sổ trượt**). Mỗi vị trí nhận một điểm số tin cậy thô, được ghi lại tại tâm của cửa sổ đó. Vì các cửa sổ không thể có tâm nằm trong phạm vi `win // 2` điểm ảnh tính từ biên (một cửa sổ gần cạnh không vừa bên trong ảnh), điểm số không được tính trong một dải viền, hiển thị màu xám.

In [ ]:
def make_scene(rng, size=64, face_size=16, n_faces=2):
    scene = rng.uniform(0, 0.5, (size, size)).astype(np.float32)
    corners = []
    tries = 0
    while len(corners) < n_faces and tries < 50:
        tries += 1
        x0 = rng.integers(0, size - face_size)
        y0 = rng.integers(0, size - face_size)
        if any(abs(x0 - px) < face_size and abs(y0 - py) < face_size for px, py in corners):
            continue
        face = make_face(size=face_size, rng=rng)
        scene[y0:y0+face_size, x0:x0+face_size] = np.maximum(scene[y0:y0+face_size, x0:x0+face_size], face)
        corners.append((x0, y0))
    # report boxes by center, not top-left corner; cast to plain float so printing doesn't show np.float64(...)
    centers = [(float(x0 + face_size / 2), float(y0 + face_size / 2)) for x0, y0 in corners]
    return scene, centers

scene_rng = np.random.default_rng(21)
scene, true_boxes = make_scene(scene_rng)
print(f'true face centers: {true_boxes}')

def sliding_window_scores(model, scene, win=16, stride=1):
    half = win // 2
    scores = np.full(scene.shape, np.nan, dtype=np.float32)  # NaN = no window centered here (too close to the border)
    with torch.no_grad():
        for y0 in range(0, scene.shape[0] - win + 1, stride):
            for x0 in range(0, scene.shape[1] - win + 1, stride):
                patch = scene[y0:y0+win, x0:x0+win]
                scores[y0 + half, x0 + half] = model(torch.tensor(patch[None, None]).float()).item()
    return scores

score_map = sliding_window_scores(model, scene)

cmap = plt.cm.hot.copy()
cmap.set_bad(color='gray')  # border pixels with no score (NaN) render as gray, not as an arbitrary color-scale value

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(scene, cmap='gray')
for cx, cy in true_boxes:
    axes[0].add_patch(patches.Rectangle((cx - 8, cy - 8), 16, 16, edgecolor='lime', facecolor='none', linewidth=2))
axes[0].set_title('scene (true faces in green)'); axes[0].axis('off')
im = axes[1].imshow(score_map, cmap=cmap)
axes[1].set_title('window classifier score, by window center\n(gray border = no window fits there)'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.show()

Ảnh động bên dưới cho thấy cửa sổ trượt qua ảnh, từng hàng một, xây dựng bản đồ nhiệt điểm số trong quá trình đó — kết thúc với cùng bản đồ điểm số như ở trên.

<img src="../animations/sliding_window_scan_demo.gif" alt="Ảnh động quét cửa sổ trượt: một khung cửa sổ vàng quét từng hàng qua cảnh trong khi bản đồ nhiệt điểm số bên phải được điền vào tương ứng, hội tụ về hai điểm nóng có tâm tại hai vị trí khuôn mặt thực" style="display: block; margin: 0 auto; max-width: 820px;">

## Bước 3: phân ngưỡng, rồi gộp các trùng lặp

Bản đồ điểm số đạt đỉnh gần các khuôn mặt thực, nhưng việc phân ngưỡng tạo ra một *cụm* phát hiện quanh mỗi khuôn mặt — mọi cửa sổ chồng lên một khuôn mặt đủ nhiều đều có điểm số trên ngưỡng. Đây chính là bài toán phát hiện trùng lặp mà Bài 12 lần đầu nêu ra cho cả Canny lẫn biến đổi Hough: nhiều giả thuyết gần giống hệt nhau cần thu gọn thành một. Cách khắc phục là **triệt tiêu điểm không cực đại (non-maximum suppression — NMS)**: lặp lại việc giữ phát hiện có điểm số cao nhất còn lại và loại bỏ mọi phát hiện khác chồng lấn với nó vượt quá một ngưỡng IoU (intersection-over-union — tỉ số giao trên hợp).

In [ ]:
def iou(a, b):
    acx, acy, aw, ah = a[:4]; bcx, bcy, bw, bh = b[:4]
    ax0, ay0, ax1, ay1 = acx - aw / 2, acy - ah / 2, acx + aw / 2, acy + ah / 2
    bx0, by0, bx1, by1 = bcx - bw / 2, bcy - bh / 2, bcx + bw / 2, bcy + bh / 2
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
    inter = iw * ih
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0

def nms(detections, iou_thresh=0.3):
    dets = sorted(detections, key=lambda d: -d[4])
    keep = []
    while dets:
        best = dets.pop(0)
        keep.append(best)
        dets = [d for d in dets if iou(best, d) < iou_thresh]
    return keep

# a threshold set from the background score distribution: well above typical background,
# well below the score at a well-aligned face window
valid = ~np.isnan(score_map)
threshold = np.percentile(score_map[valid], 97.5)
raw_detections = [(cx, cy, 16, 16, score_map[cy, cx])  # (cx, cy) is already the window's center
                   for cy, cx in zip(*np.where(valid)) if score_map[cy, cx] > threshold]
final_detections = nms(raw_detections)

print(f'threshold (97.5th percentile of all window scores): {threshold:.2f}')
print(f'raw detections above threshold: {len(raw_detections)}')
print(f'detections after NMS: {len(final_detections)}')
for cx, cy, w, h, s in final_detections:
    print(f'  box=(cx={cx:.0f}, cy={cy:.0f}, w={w}, h={h})  score={s:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(scene, cmap='gray')
for cx, cy in true_boxes:
    ax.add_patch(patches.Rectangle((cx - 8, cy - 8), 16, 16, edgecolor='lime', facecolor='none', linewidth=3, label='ground truth'))
for cx, cy, w, h, s in final_detections:
    ax.add_patch(patches.Rectangle((cx - w / 2, cy - h / 2), w, h, edgecolor='red', facecolor='none', linewidth=1.5, linestyle='--', label='detection'))
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='upper right')
ax.set_title(f'{len(final_detections)} detections after NMS vs. {len(true_boxes)} true faces')
ax.axis('off')
plt.show()

Ảnh động bên dưới chạy triệt tiêu điểm không cực đại từng bước: bắt đầu từ 60 phát hiện thô (cam) trên ngưỡng, mỗi vòng chọn hộp có điểm số cao nhất còn lại (đỏ), triệt tiêu mọi thứ chồng lấn với nó quá nhiều (xám), và giữ nó lại (đỏ) — lặp lại cho đến khi chỉ còn các hộp cuối cùng ở trên. Các hộp chuẩn (xanh lục) được hiển thị để so sánh.

<img src="../animations/nms_demo.gif" alt="Ảnh động triệt tiêu điểm không cực đại: bắt đầu từ 60 phát hiện thô tụ quanh hai khuôn mặt, mỗi vòng làm nổi bật hộp có điểm số cao nhất màu đỏ, đánh dấu mọi thứ nó triệt tiêu màu xám, và giữ nó lại màu đỏ, thu gọn các cụm xuống còn 2 phát hiện cuối cùng khớp sát với 2 vị trí khuôn mặt thực (xanh lục)" style="display: block; margin: 0 auto; max-width: 620px;">

Với ngưỡng này và `iou_thresh=0.3`, NMS thu gọn các phát hiện thô xuống còn hai hộp khớp sát với hai vị trí khuôn mặt thực (một hộp rơi đúng vào một tâm thực, hộp kia lệch trong vòng vài điểm ảnh). Như các bài tập bên dưới khám phá, kết quả trên các cảnh và seed khác cho ra dương tính giả và âm tính giả, tùy thuộc vào ngưỡng và các lựa chọn tham số khác.

## Xử lý tỉ lệ: kim tự tháp ảnh

Mọi thứ ở trên đều giả định khuôn mặt luôn có kích thước chính xác 16x16, nhưng khuôn mặt thực xuất hiện ở các tỉ lệ chưa biết. Giải pháp là kim tự tháp Gauss ở Bài 11 (`cv2.pyrDown`): dựng một chồng ảnh ở nhiều độ phân giải, và chạy *cùng* bộ phân loại cửa sổ kích thước cố định trên mọi tầng. Một khuôn mặt quá lớn so với cửa sổ ở độ phân giải đầy đủ sẽ vừa với cửa sổ ở một tầng kim tự tháp thô hơn nào đó, vì thu nhỏ ảnh tương đương với phóng to kích thước cửa sổ hiệu dụng so với nội dung ảnh. Cảnh của bài này chỉ có một tỉ lệ khuôn mặt, nên kim tự tháp không được minh họa trực tiếp ở đây — nhưng cơ chế là như nhau.

## Viola-Jones

Bộ phân loại cửa sổ Rowley-Baluja-Kanade (thứ bài này vừa xây dựng, ở dạng thu nhỏ) nặng về tính toán vào thời điểm đó. Năm 1996 (trước thời GPU), đánh giá một mạng nơ-ron nhỏ ở mọi vị trí và tỉ lệ của mọi tầng kim tự tháp là chậm. <a href="../references.html#viola-jones-2001">Viola và Jones (2001)</a> đã sửa đổi cùng ý tưởng cửa sổ trượt để chạy thời gian thực bằng cách thay mạng nơ-ron bằng một **chuỗi tầng (cascade)** các đặc trưng kiểu Haar cực kỳ rẻ (cùng ý tưởng tổng/hiệu cục bộ như wavelet Haar ở Bài 16, áp dụng cho các vùng hình chữ nhật 2D): một chuỗi các giai đoạn, mỗi giai đoạn là một ngưỡng đơn giản trên hiệu cường độ của một vùng hình chữ nhật, được sắp xếp sao cho tuyệt đại đa số cửa sổ không phải khuôn mặt bị loại bỏ ngay ở giai đoạn *đầu tiên* hoặc thứ hai, và chỉ những cửa sổ hiếm hoi đầy hứa hẹn mới phải trả giá cho toàn bộ chuỗi bộ phân loại. Kết quả là, chi phí trung bình của chuỗi tầng cho mỗi cửa sổ rất nhỏ — đó là lý do Viola-Jones là thuật toán cuối cùng được chạy trực tiếp trên các máy ảnh kỹ thuật số thời những năm 2000.

OpenCV cung cấp chuỗi tầng Viola-Jones dưới dạng `cv2.CascadeClassifier`, đã được tiền huấn luyện trên khuôn mặt thật. Chỉ cần tải một tệp XML chứa các giai đoạn chuỗi tầng đã học và chạy nó.

In [ ]:
import urllib.request
from pathlib import Path
import cv2

CACHE_DIR = Path.home() / '.cache' / 'cvintro'
CASCADE_URL = 'https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml'
CASCADE_PATH = CACHE_DIR / 'haarcascade_frontalface_default.xml'

def ensure_cascade():
    if CASCADE_PATH.exists():
        return
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading the Viola-Jones frontal-face cascade (one-time, cached under ~/.cache/cvintro)...')
    urllib.request.urlretrieve(CASCADE_URL, CASCADE_PATH)

ensure_cascade()

cascade = cv2.CascadeClassifier(str(CASCADE_PATH))
photo = cv2.imread('../img/apollo11_crew.jpg')
gray = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)

faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
print(f'{len(faces)} detections')

photo_rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(photo_rgb)
for (x, y, w, h) in faces:
    ax.add_patch(patches.Rectangle((x, y), w, h, edgecolor='lime', facecolor='none', linewidth=2))
ax.set_title(f'cv2.CascadeClassifier: {len(faces)} detections')
ax.axis('off')
plt.show()

<p class="image-attribution">Nguồn ảnh: NASA (phạm vi công cộng), qua <a href="https://commons.wikimedia.org/wiki/File:Apollo_11_Crew.jpg" target="_blank" rel="noopener">Wikimedia Commons</a></p>

Khuôn mặt của cả ba phi hành gia (Armstrong, Collins, Aldrin) đều được tìm thấy chính xác, cộng với một dương tính giả nhỏ trên kết cấu vải ở tay áo của Aldrin. `minNeighbors` (một vùng phải có bao nhiêu phát hiện chồng lấn) và `scaleFactor` (kim tự tháp ảnh bước giữa các tỉ lệ thô đến đâu) đánh đổi giữa dương tính giả và khuôn mặt bị bỏ sót, đóng vai trò tương tự như ngưỡng NMS của bài này.

### Bài tập

1. Đổi `iou_thresh` trong `nms` từ `0.3` sang `0.7`. Chạy lại phát hiện trên cảnh. NMS giờ gộp thiếu (báo cáo nhiều hơn 2 hộp) hay gộp thừa (bỏ sót một khuôn mặt)? Hãy giải thích dựa trên mức độ chồng lấn thực tế giữa các phát hiện trùng lặp quanh cùng một khuôn mặt.
2. Ngưỡng ở đây được đặt từ phân vị thứ 97.5 của phân phối điểm số của *chính cảnh này* — một hình thức gian lận, vì một bộ phát hiện thực tế không được xem điểm số của cảnh kiểm tra trước khi quyết định. Thay vào đó, hãy tính một ngưỡng chỉ từ các điểm số khuôn mặt/không-phải-khuôn-mặt đã biết của `Xte` (ví dụ, điểm giữa của điểm số khuôn-mặt-thực thấp nhất và điểm số không-phải-khuôn-mặt-thực cao nhất), và kiểm tra xem nó có còn phát hiện thành công cả hai khuôn mặt trong cảnh không.
3. Tăng `n_faces` trong `make_scene` lên 4 và thu nhỏ `size` xuống 48 để các khuôn mặt xếp gần nhau hơn. NMS có còn tách chúng đúng không, hay phép triệt tiêu dựa trên IoU bắt đầu gộp các khuôn mặt thực sự riêng biệt ở gần nhau thành một phát hiện? Ở khoảng cách nào thì nó thất bại?
4. Hạ `minNeighbors` trên chuỗi tầng Viola-Jones thật từ `5` xuống `2` và chạy lại. Dương tính giả trên vải có tệ hơn không, hay các dương tính giả mới xuất hiện ở chỗ khác? Sau đó hãy thử tăng `minNeighbors` lên `8` — nó có dọn sạch dương tính giả không, và nó có làm mất phát hiện đúng nào không?